In [267]:
import os, sys
import numpy as np
import math

sys.path.append(os.path.abspath(os.path.join('..', '..')))


from collections import defaultdict, Counter
from dotenv import load_dotenv

from myapp.search import load_corpus as lc
from project_progress.part_1.data_prep import build_terms, join_build_terms

load_dotenv()  # take environment variables from .env

True

In [268]:
def corpus_df_loading(path):
    """
    In this function we load the corpus as dataframe, and we preprocess the numerical fields.

    :param path: Path to the json file.
    :return corpus: Returns a dictionary List[Document] with the loaded corpus with the numerical fields preprocessed.
    """
    corpus = lc.load_corpus(path)
    return corpus

In [269]:
def create_index_tf_idf(corpus):
    """
    Implement the inverted index and compute tf, df and idf

    Argument:
    corpus -- 

    #TODO: adapt to our version

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of document these keys appears in (and the positions) as values.
    index2title - a mapping of article pid to its title
    tf - normalized term frequency for each term in each document
    df - number of documents each term appear in
    idf - inverse document frequency of each term
    """

    index = defaultdict(dict)
    tf = defaultdict(dict)
    df = defaultdict(dict)
    idf = defaultdict(dict)
    index2title = {}
    num_articles = len(corpus)

    for doc in list(corpus.values()):
        index2title[doc.pid] = doc.title
        # For each field to be considered in the index, get its terms
        title_description = join_build_terms([doc.title, doc.description])  # Pre-process the `title` and `description`
        brand_terms = join_build_terms([doc.brand])
        category_terms = join_build_terms([doc.category])
        sub_category_terms = join_build_terms([doc.sub_category])
        seller_product_details = join_build_terms([doc.seller, " ".join([detail for detail in doc.product_details.values()])])  # Pre-process the `title` and `description`

        # Fields and target terms to process
        fields = ['title_description', 'brand', 'category', 'sub_category', 'seller_product_details']
        target = [title_description, brand_terms, category_terms, sub_category_terms, seller_product_details]

        # Initialize a temporal dictionary to store the index terms for the current article
        current_article_index = defaultdict(dict)

        # Create the index for the current article
        # For each field we consider in the index
        for idx, field in enumerate(fields):    
            # For each term in the target field
            for position, term in enumerate(target[idx]):   
                try:
                    # Add the new found term's position to the dict
                    current_article_index[term][field][1].append(position)  
                except:
                    # Create the entry for the term and field with the term's position if it didn't exist
                    current_article_index[term][field] = [doc.pid, [position]]  

        for field in fields:
            norm = 0
            for term, positions in current_article_index.items():
                norm += len(positions[field][1])**2
            norm = math.sqrt(norm)

            for term, positions in current_article_index.items():
                # Compute term frequency and document frequency of each term per category
                try:
                    tf[term][field].append(np.round(len(positions[field][1])/norm, 4))
                    df[term][field] += 1
                # If it's a term we haven't seen, create a new term frequency and document frequency entry
                except:
                    tf[term][field] = [np.round(len(positions[field][1])/norm, 4)]
                    df[term][field] = 1

                # In the practice, the tf/df and the index were in separate loops. Both codes are now in one 
                # loop to avoid reading the same twice
                # Join the current article's index with the global index
                try:
                    index[term][field].append(positions[field])  # Add the array of positions ("[id, [[0],[1]]]"") in the given term and field
                except:
                    index[term][field] = [positions[field]]      # Create the entry for the term and the field with the array of positions


    for term, posting_fields in df:
        for field, posting in posting_fields:
            idf[term][field] = np.round(np.log(float(num_articles / df[term])), 4)

    return index, index2title, tf, df, idf

In [270]:
json_path = "../../data/fashion_products_dataset.json"
corpus = corpus_df_loading(json_path)

In [254]:
print(corpus.keys())

dict_keys(['TKPFCZ9EA7H5FYZH', 'TKPFCZ9EJZV2UVRZ', 'TKPFCZ9EHFCY5Z4Y', 'TKPFCZ9ESZZ7YWEF', 'TKPFCZ9EVXKBSUD7', 'TKPFCZ9EFK9DNWDA', 'TKPFDABN3GXYPFHE', 'TKPFCZ9ESGZYT8NH', 'TKPFCZ9DYU33FFXS', 'TKPFDABN4NQFVKZY', 'TKPFCZ9ENWGMX23W', 'TKPFZFSHHACG3FHC', 'TKPFZFSHQPDRGZTM', 'TKPFZ4YTRF3ZRTTH', 'TKPFZ4YTJZWBFYFZ', 'TKPFZFSH3F9ZA7C6', 'TKPFZ4YT7ZNYXG27', 'TKPFZ4YTGNZJDZDU', 'TKPFZ4YTX94CY9JX', 'TKPFCZ9EHCNAPKPU', 'TKPFDACEXAWUHGR7', 'TKPFCZ9ETR6YVXNG', 'TKPFD3K6K5TNYZGF', 'TKPFCZ9EGGYENTZS', 'TKPFD3K6ZMN79MPH', 'TKPFD3K6UZBYDZNY', 'TKPFD3K62JB9PEMR', 'TKPFCZ9EZDPZR5AH', 'TKPFWBGVGU9FCAYX', 'TKPFWBGPGVKVZARU', 'TKPFCZ9EVM2GZ4GF', 'TKPFWAG7YFWPMG5Y', 'TKPFCZ9E2UC3DR3F', 'TKPFCZ9ECDYYDNKA', 'SOCFVCCYUDEN2AVH', 'CTPFVZTEMJWEJJJV', 'CTPFVQP2PEHKGYFQ', 'CTPFVPM3NDPBPPXE', 'CTPFVZHSA7G4PFC5', 'CTPFVZD8CNSZ3AMR', 'CTPFVQNNHGYFTGFN', 'CTPFVZT3UFN99ZTH', 'CTPFVZHY42MSZCF6', 'CTPFVSU7CXFCXEHD', 'CTPFVZGRKPGSFPUU', 'CTPFVZT7EFZWVRUP', 'CTPFVXGGEHH6YY8G', 'CTPFVQZFE4HTFZNG', 'CTPFVPKJGJBXHQFJ', 'CTPFVZT2

In [255]:
def create_index_tf_idf(corpus):
    """
    Implement the inverted index and compute tf, df and idf

    Argument:
    corpus --

    #TODO: adapt to our version

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of document these keys appears in (and the positions) as values.
    index2title - a mapping of article pid to its title
    tf - normalized term frequency for each term in each document
    df - number of documents each term appear in
    idf - inverse document frequency of each term
    """

    index = defaultdict(dict)
    tf = defaultdict(lambda: defaultdict(dict))
    df = defaultdict(int)
    idf = {}
    index2title = {}
    num_articles = len(corpus)

    for product in list(corpus.values()):
        index2title[product.pid] = product.title
        # For each field to be considered in the index, get its terms
        title_description = join_build_terms(
            [product.title, product.description]
        )  # Pre-process the `title` and `description`
        brand_terms = join_build_terms([product.brand])
        category_terms = join_build_terms([product.category])
        sub_category_terms = join_build_terms([product.sub_category])
        seller_product_details = join_build_terms(
            [product.seller, " ".join([detail for detail in product.product_details.values()])]
        )  # Pre-process the `title` and `description`

        # Fields and target terms to process
        fields = [
            "title_description",
            "brand",
            "category",
            "sub_category",
            "seller_product_details",
        ]
        target = [
            title_description,
            brand_terms,
            category_terms,
            sub_category_terms,
            seller_product_details,
        ]

        # Initialize a temporal dictionary to store the index terms for the current article
        current_article_index = defaultdict(dict)

        # Create the index for the current article
        # For each field we consider in the index
        for field_idx, field in enumerate(fields):
            # For each term in the target field
            for position, term in enumerate(target[field_idx]):
                try:
                    # Add the new found term's position to the dict
                    current_article_index[term][field][1].append(position)
                except:
                    # Create the entry for the term and field with the term's position if it didn't exist
                    current_article_index[term][field] = [product.pid, [position]]

        seen_terms = set()
        for field in fields:
            norm = 0
            for term, postings in current_article_index.items():
                if field in postings.keys():    # Check that the term appear in the current field
                    norm += len(postings[field][1]) ** 2
            norm = math.sqrt(norm)

            for term, postings in current_article_index.items():
                if field in postings.keys():
                    pid = postings[field][0]
                    # Compute term frequency and document frequency of each term per category
                    tf[pid][term][field] = np.round(len(postings[field][1]) / norm, 4)
                    

                    # In the practice, the tf/df and the index were in separate loops. Both codes are now in one
                    # loop to avoid reading the same twice
                    # Join the current article's index with the global index
                    try:
                        index[term][field].append(
                            postings[field]
                        )  # Add the array of positions ("[id, [[0],[1]]]"") in the given term and field
                    except:
                        index[term][field] = [
                            postings[field]
                        ]  # Create the entry for the term and the field with the array of positions

                if term not in seen_terms:
                    df[term] += 1 
                    seen_terms.add(term)
        


    for term in df.keys():
        idf[term] = (
            np.round(np.log(float(num_articles / df[term])), 4)
        )

    return index, index2title, tf, df, idf


In [256]:
index, index2title, tf, df, idf = create_index_tf_idf(corpus=corpus)

In [257]:
def filter(query, index):
    """
    The output is the list of documents that contain ALL query terms.

    :param query: (string) query
    :param index: (Dict) inverted index dictinary
    :return selected_docs: (List) of documents' ids that contain all query terms
    """

    query_terms = build_terms(query)
    docs = None
    for term in query_terms:
        try:
            # Get all doc ids from the term
            all_doc_ids = [
                doc_id
                for categories in index[term].values()
                for doc_id, _ in categories
            ]

            # Get intersection of documents with ALL the terms
            if docs is None:  # First time, set is empty
                docs = set(all_doc_ids)  # Initiallize with first term's doc ids
            else:
                docs &= set(all_doc_ids)

        except:
            pass

    return query_terms, list(docs)

In [258]:
type(index2title)

dict

In [259]:
print(list(corpus.items())[:2])

[('TKPFCZ9EA7H5FYZH', Document(pid='TKPFCZ9EA7H5FYZH', title='Solid Women Multicolor Track Pants', description='Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India', brand='York', category='Clothing and Accessories', sub_category='Bottomwear', product_details={'Style Code': '1005COMBO2', 'Closure': 'Elastic', 'Pockets': 'Side Pockets', 'Fabric': 'Cotton Blend', 'Pattern': 'Solid', 'Color': 'Multicolor'}, seller='Shyam Enterprises', out_of_stock=False, selling_price=921.0, discount=69.0, actual_price=2999.0, average_rating=3.9, url='https://www.flipkart.com/yorker-solid-men-multicolor-track-pants/p/itmd2c76aadce459?pid=TKPFCZ9EA7H5FYZH&lid=LSTTKPFCZ9EA7H5FYZHVYXWP0&marketplace=FLIPKART&srno=b_1_1&otracker=browse&fm=organic&iid=177a46eb-d053-4732-b3de-fcad6ff59cbd.TKPFCZ9EA7H5FYZH.SEARCH&ssid=utkd4t3gb40000001612415717799', images=['https://rukminim1.fl

In [265]:
query = "solid women multicolor track pant yorker"
query_terms, products = filter(query=query, index=index)

weights = {
        "title_description": 0.5,
        "brand": 0.05,
        "category": 0.2,
        "sub_category": 0.2,
        "seller_product_details": 0.05,
    } 

In [261]:
def rank_tf_idf(query_terms, products, index, tf, idf, weights):
    """
    Perform the ranking of the results of a search based on the tf-idf weights

    Argument:
    terms -- list of query terms
    products -- list of products to rank that match the query
    index -- inverted index data structure
    tf -- term frequencies
    idf -- inverted document frequencies
    weights -- weights to average the fields to compute the rank

    Returns:
    Print the list of product ids of the ranked articles
    """
    if len(products) > 0:
        # For the docs, take only the components for the query terms 
        product_vectors = defaultdict(lambda: [0]*len(query_terms))
        query_vector = [0] * len(query_terms)


        # Compute the norm for the query tf
        query_term_counts = Counter(query_terms)
        query_norm = np.linalg.norm(list(query_term_counts.values()))

        # Compute tf-idf for each document and query
        for term_idx, q_term in enumerate(query_terms):
            if q_term not in index:
                continue

            # tf*idf (normalize TF)
            query_vector[term_idx] = (query_term_counts[q_term]/query_norm) * idf[q_term]

            # Compute the document vectors
            second_loop = False
            for field, postings in index[q_term].items():
                for pid, positions in postings:
                    if pid in products:
                        product_vectors[pid][term_idx] += tf[pid][q_term][field]* weights[field]
            
        product_scores = [[np.dot(current_prod_vec, query_vector), prod] for prod, current_prod_vec in product_vectors.items()]
        product_scores.sort(reverse=True)

        return product_scores
    else:
        return None, None

In [262]:
def engine_search(query, index):
    query_terms, filtered_docs = filter(query, index)
    scores = rank_tf_idf(query_terms, filtered_docs, index, tf, idf, weights)
    return scores

Find the most common words in the document with `tf`:

In [263]:
summed_tf = defaultdict(float)

list_tf = list(tf.items())
for doc, doc_tf in list_tf:
    for term, freq in doc_tf.items():
        summed_tf[term] += sum(float(v) for v in freq.values())

summed_tf = dict(summed_tf)

sorted_tf = dict(sorted(summed_tf.items(), key=lambda x: x[1], reverse=True))

for term, total in sorted_tf.items():
    print(f"{term:15} -> {total:.3f}")

cloth           -> 20801.618
accessori       -> 20489.974
topwear         -> 15310.490
neck            -> 7992.599
wear            -> 6392.516
cotton          -> 5983.263
women           -> 5720.502
men             -> 5621.230
round           -> 5588.518
solid           -> 5133.891
print           -> 5050.661
1               -> 4882.042
regular         -> 4652.134
sleev           -> 4637.967
wash            -> 4388.736
shirt           -> 3817.614
bottomwear      -> 3672.289
western         -> 3441.874
blend           -> 3011.922
machin          -> 2975.271
fit             -> 2713.045
black           -> 2700.837
blue            -> 2632.673
casual          -> 2444.519
india           -> 2399.398
slim            -> 2249.191
full            -> 2201.013
polo            -> 1979.192
winter          -> 1908.200
footwear        -> 1861.156
fabric          -> 1646.244
half            -> 1630.918
multicolor      -> 1482.033
2               -> 1461.472
white           -> 1453.865
short           -

In [266]:
queries = ["western leather jacket men",
           "cotton innerwear man",
           "yellow black t-shirt women xl",   
           "casual comfortable blue trousers women",         
           "breathable sports clothes winter"]         

for query in queries:
    scores = engine_search(query, index)

    print(f"QUERY: '{query}'")
    print(f"RANKING:")
    if scores[0] != None:   # there is no document as a result
        for idx, (score, pid) in enumerate(scores):
            print(f"{idx+1}. [{score:.3f}] {[index2title[pid]]}")
    else:
        print("No results.")
    print("-------------------------------------------------------------------------------------------------------------------\n")

QUERY: 'western leather jacket men'
RANKING:
1. [0.963] ['Full Sleeve Solid Men Leather Jacket']
2. [0.963] ['Full Sleeve Solid Men Leather Jacket']
3. [0.963] ['Full Sleeve Solid Men Leather Jacket']
4. [0.963] ['Full Sleeve Solid Men Leather Jacket']
5. [0.963] ['Full Sleeve Solid Men Leather Jacket']
6. [0.780] ['Full Sleeve Solid Men Leather Jacket']
7. [0.780] ['Full Sleeve Solid Men Leather Jacket']
8. [0.780] ['Full Sleeve Solid Men Leather Jacket']
9. [0.703] ['Full Sleeve Color Block, Applique Men Leather Jacket']
10. [0.703] ['Full Sleeve Color Block, Applique Men Leather Jacket']
11. [0.688] ['Full Sleeve Solid Men Leather Jacket']
12. [0.566] ['Full Sleeve Solid Men Quilted Jacket']
13. [0.547] ['Full Sleeve Printed Men Denim Casual Jacket']
14. [0.503] ['Full Sleeve Washed Men Denim Jacket']
15. [0.503] ['Full Sleeve Washed Men Denim Jacket']
16. [0.499] ['Full Sleeve Solid Men Jacket']
17. [0.497] ['Full Sleeve Solid Men Jacket']
18. [0.496] ['Full Sleeve Washed Men Denim